# SmartAd Performance Analysis — Sequential A/B Testing (SPRT)

---

## 📋 Overview

This notebook implements **Sequential Probability Ratio Testing (SPRT)** — an alternative to
classical A/B testing that allows continuous data monitoring with early stopping.

### Why Sequential Testing?

Classical tests require you to **pre-specify the sample size** before the experiment begins.
Peeking at results early and stopping when significance is reached inflates the Type I error rate.

Sequential tests solve this by providing **valid statistical guarantees at every observation point**:

| Classical A/B Test | Sequential SPRT |
|---|---|
| Fixed sample size pre-determined | No pre-specified sample size needed |
| Cannot peek at results mid-test | Designed for continuous monitoring |
| Decision made once at the end | Decision made as soon as evidence is conclusive |
| Wastes data if effect is large | Stops early if effect is large (saves cost) |

---

## 🔬 Mathematical Framework: Meeker's Conditional SPRT

This implementation uses **Meeker's Conditional SPRT** for matched binary sequences.

### Decision Rule

At each observation $n$, compute the cumulative log-likelihood ratio $S_n$:

$$S_n = \sum_{i=1}^{n} g(x_i, r_i, n_i, t_1)$$

where $g$ is the log probability ratio between H₁ (odds ratio $t_1$) and H₀ (odds ratio 1).

Compare $S_n$ against two boundaries:

| Condition | Decision |
|---|---|
| $S_n \geq U = -\log(\alpha / (1-\beta))$ | **Reject H₀**: Significant lift detected |
| $S_n \leq L = \log(\beta / (1-\alpha))$ | **Accept H₀**: No significant difference |
| $L < S_n < U$ | **Continue**: Collect more data |

### Parameters

| Parameter | Description | Default |
|---|---|---|
| $\alpha$ | Type I error rate (false positive probability) | 0.05 |
| $\beta$ | Type II error rate (false negative probability) | 0.10 |
| $t_1$ | Odds ratio under H₁ (expected effect size) | 2.0 |

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'script'))

from sequential_test_script import (
    conditionalSPRT,
    transform_data,
    get_bernoulli_series,
    plot_sprt,
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.family': 'DejaVu Sans', 'figure.dpi': 120})
print('✅ Imports successful')

---
## 📂 1. Load Data and Transform to Bernoulli Sequences

In [ ]:
# Load raw data
df_raw = pd.read_csv('../data/AdSmart_AB_testing_data.csv')
print(f'Raw shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
# Transform raw data into matched Bernoulli sequences
# Steps:
#   1. Remove non-respondents (yes=0, no=0)
#   2. Split into exposed and control groups
#   3. Aggregate by date_hour
#   4. Build shuffled Bernoulli series (1=aware, 0=not aware)

control_arr, exposed_arr = transform_data(df_raw)

print(f'Control Bernoulli sequence length  : {len(control_arr):,}')
print(f'Exposed Bernoulli sequence length  : {len(exposed_arr):,}')
print(f'Control success rate               : {control_arr.mean():.4f}')
print(f'Exposed success rate               : {exposed_arr.mean():.4f}')
print(f'Observed lift                      : {exposed_arr.mean() - control_arr.mean():+.4f}')

---
## 🧪 2. Run the Conditional SPRT

We run Meeker's Conditional SPRT with:
- α = 0.05 (5% Type I error tolerance)
- β = 0.10 (10% Type II error tolerance)
- t₁ = 2.0 (we expect the exposed group to have 2× the odds of brand awareness)
- Maximum 800 observations as a truncation safeguard

In [ ]:
# Run SPRT
result = conditionalSPRT(
    x=exposed_arr,     # Treatment: exposed group Bernoulli series
    y=control_arr,     # Control group Bernoulli series
    t1=2.0,            # Odds ratio under H₁
    alpha=0.05,
    beta=0.10,
    stop=800,          # Truncation point
)

print('='*60)
print(f"Outcome           : {result['outcome']}")
print(f"Decision at n     : {result['decision_index']}")
print(f"Total observations: {result['n_observations']}")
print(f"Lower bound (L)   : {result['lower_bound']:.4f}")
print(f"Upper bound (U)   : {result['upper_bound']:.4f}")
print(f"Truncation        : {result['truncated']}")
print('='*60)

---
## 📊 3. Visualise the Decision Trajectory

The SPRT plot shows:
- **Observed cumulative successes** in the exposed group (trajectory line)
- **Upper decision boundary** (reject H₀: significant lift detected)
- **Lower decision boundary** (accept H₀: no significant difference)
- **Decision star** marking the observation where the boundary was crossed

In [ ]:
# Plot SPRT decision trajectory
plot_sprt(result)

---
## 📈 4. Sensitivity Analysis: Effect of Odds Ratio

How does the assumed odds ratio t₁ affect when (and whether) a decision is reached?

In [ ]:
import math

t1_values = [1.5, 2.0, 2.5, 3.0, 4.0]
results_summary = []

for t1 in t1_values:
    res = conditionalSPRT(exposed_arr, control_arr, t1=t1, alpha=0.05, beta=0.10, stop=800)
    k = res['decision_index']
    k_disp = str(k) if not (isinstance(k, float) and math.isnan(k)) else 'No decision'
    results_summary.append({
        'Odds Ratio (t₁)': t1,
        'Decision at n': k_disp,
        'Outcome': res['outcome'][:60] + '...' if len(res['outcome']) > 60 else res['outcome'],
        'L boundary': round(res['lower_bound'], 4),
        'U boundary': round(res['upper_bound'], 4),
    })

pd.DataFrame(results_summary)

---
## 🏁 5. Conclusions

### Sequential vs Classical — Decision Comparison

| Method | Sample Used | Decision | Cost Efficiency |
|---|---|---|---|
| Classical Z-test | Full dataset | Based on final n | Standard |
| SPRT (t₁=2.0) | Up to decision point | Based on trajectory | Better if early stop |

### Key Advantages of SPRT for SmartAd

1. **Cost savings**: If the ad shows a strong effect early, the test can be stopped early,
   reducing the number of impressions needed for a decision.

2. **Valid continuous monitoring**: Unlike classical tests, SPRT maintains valid error
   rate control even when checking the data at every observation.

3. **Flexible**: The truncation parameter allows setting a maximum budget for the test.

### Limitations

- The odds ratio t₁ must be chosen in advance based on domain knowledge.
- If the true effect is much smaller than t₁, the test may never reach a decision.
- Sequential tests are more computationally intensive than simple Z-tests.

**Next step**: See `ml_AB_testing.ipynb` for a Machine Learning approach that identifies
*which user segments* are most responsive to the SmartAd creative.